# 11 · A foundation model's picture of each place: TESSERA

**Question:** does adding TESSERA's description of each 100 m cell at the window's start improve the improved risk model (notebooks 05–07)?

**TESSERA** (Feng et al. 2026) reads a full year of Sentinel-1 radar and Sentinel-2 imagery for every 10 m pixel and compresses it into 128 numbers. Its authors publish these yearly embeddings for the whole globe from 2017, free under CC0. Averaged over a 100 m cell, the vector of a year describes the cell as it was that year: how much of it is closed canopy, fallow, crops or bare ground, and how it behaved through the seasons. It is used here as one more predictor measured at the window's start, as every predictor of the improved model is (notebook 05).

**How it was tested** (`scripts/tessera_test.py`, runs `outputs/runs/11_tessera_*.json`):

* **Blocks.** All of Guinea is 300–370 GB of embeddings a year, so the test uses 16 forest blocks of 0.1° (about 11 × 11 km, 12,000 cells each), TESSERA's own tiles: tiles at least 30 % forest in 2000, shared among the forest ecoregions in proportion to their 2018–2024 loss and drawn with probability proportional to the tile's loss (`data_100m_tools/tessera_blocks.py`). Each cell gets the mean of its 10 × 10 pixels (`data_100m_tools/tessera_fetch.py`; checked against direct reads, identical).
* **Windows.** TESSERA starts in 2017 and Hansen in the store ends in 2024, so two windows have an embedding strictly before the calibration loss: calibration 2018–2020 with predictors at 2017, test 2021–2023 with predictors at 2020; and calibration 2019–2021 (2018), test 2022–2024 (2021), the third window of notebooks 03–10.
* **Models.** The improved model's random forest and draw of training cells (notebook 04), seeds 42, 1 and 2, fitted and scored on the same cells in the blocks: the 16 predictors; with the 128 numbers; with their first 16 principal components; the 128 numbers alone. AUC over the 30 m pixels, as notebook 04. The gain's 90 % interval comes from resampling the 16 blocks (2,000 draws).

In [1]:
import json
from pathlib import Path
import pandas as pd

RUNS = Path.cwd().parent / "outputs" / "runs"
SPLITS = [(2017, 2020, 2023), (2018, 2021, 2024)]
main = [json.loads((RUNS / "11_tessera_{}_{}_{}.json".format(*s)).read_text()) for s in SPLITS]
checks = [json.loads((RUNS / "11_tessera_checks_{}_{}_{}.json".format(*s)).read_text()) for s in SPLITS]
esri = [json.loads((RUNS / "11_tessera_esri_{}_{}_{}.json".format(*s)).read_text()) for s in SPLITS]
blocks = json.loads(Path(r"E:\SDG AI Lab\Guinea\data_100m\tessera\blocks.json").read_text())["blocks"]
pd.DataFrame([b for b in blocks if b["kind"] == "forest"]).drop(columns=["kind"])

,name,bbox,ecoregion,cells,forest2000_pct,loss_2018_2024_ha
0,-11.7_10.3,"[-11.7, 10.3, -11.6, 10.4]",Guinean forest-savanna,12321,90.4,1215.4
1,-13.4_11.3,"[-13.4, 11.3, -13.3, 11.4]",Guinean forest-savanna,12432,51.5,1996.5
2,-14.3_10.6,"[-14.3, 10.6, -14.2, 10.7]",Guinean forest-savanna,12432,68.3,2009.9
3,-14.6_11.1,"[-14.6, 11.1, -14.5, 11.2]",Guinean forest-savanna,12544,50.9,1172.2
4,-10.2_9.7,"[-10.2, 9.7, -10.1, 9.8]",Guinean forest-savanna,12432,39.4,886.5
5,-9.8_8.6,"[-9.8, 8.6, -9.7, 8.7]",Guinean forest-savanna,12432,97.7,2198.1
6,-12.1_10.2,"[-12.1, 10.2, -12.0, 10.3]",Guinean forest-savanna,12432,86.5,1990.4
7,-10.7_10.5,"[-10.7, 10.5, -10.6, 10.6]",Guinean forest-savanna,12321,67.7,590.6
8,-14.6_10.9,"[-14.6, 10.9, -14.5, 11.0]",Guinean mangroves,12432,73.3,1657.3
9,-9.1_8.6,"[-9.1, 8.6, -9.0, 8.7]",Guinean montane forests,12432,47.8,255.4


## 1 · Results

Mean AUC over the three seeds, and the gain over the improved model with its 90 % interval (blocks resampled). Seed spread is under 0.01 for every model.

In [2]:
rows = []
for r in main:
    for label, aucs in r["auc_per_seed"].items():
        g = r["gain_vs_improved"].get(label)
        rows.append({"Window": r["split"], "Model": label, "AUC": round(sum(aucs) / 3, 3),
                     "Seeds": ", ".join(f"{a:.3f}" for a in aucs),
                     "Gain over improved": "" if g is None else f"{g['gain']:+.3f} ({g['ci90'][0]:+.3f} to {g['ci90'][1]:+.3f})"})
pd.DataFrame(rows)

,Window,Model,AUC,Seeds,Gain over improved
0,2018–2020 → 2021–2023,Improved model,0.657,"0.668, 0.651, 0.651",
1,2018–2020 → 2021–2023,Improved + TESSERA,0.810,"0.811, 0.809, 0.809",+0.150 (+0.105 to +0.194)
2,2018–2020 → 2021–2023,Improved + TESSERA (16 PCs),0.803,"0.806, 0.801, 0.803",+0.145 (+0.107 to +0.185)
3,2018–2020 → 2021–2023,TESSERA alone,0.797,"0.796, 0.796, 0.798",+0.136 (+0.085 to +0.186)
4,2019–2021 → 2022–2024,Improved model,0.670,"0.676, 0.670, 0.665",
5,2019–2021 → 2022–2024,Improved + TESSERA,0.786,"0.787, 0.785, 0.785",+0.111 (+0.073 to +0.149)
6,2019–2021 → 2022–2024,Improved + TESSERA (16 PCs),0.788,"0.789, 0.786, 0.787",+0.113 (+0.083 to +0.147)
7,2019–2021 → 2022–2024,TESSERA alone,0.771,"0.772, 0.770, 0.769",+0.095 (+0.053 to +0.139)


In [3]:
pd.DataFrame([{"Window": r["split"], "Test cells": r["test_cells"], "Loss in the test window (ha)": round(r["lost_ha"]),
               "Variance kept by 16 components": round(r["pca_variance_16"], 2)} for r in main])

,Window,Test cells,Loss in the test window (ha),Variance kept by 16 components
0,2018–2020 → 2021–2023,163605,9075,0.87
1,2019–2021 → 2022–2024,162150,8565,0.88


## 2 · Is the gain real?

A gain 50 times that of any predictor tested in notebook 05 needs checking. Two ways it could be false:

* **Dating.** Hansen often dates a clearing to the year after it happened, so the embedding of the year before a window may already show a loss counted in the window's first year. **Gap:** the first year of both windows is left out (calibration loss from t1 + 2, test loss from t2 + 2).
* **Memory of places.** With 128 numbers a model could recognise places and find their loss again. **Other blocks:** each block is scored by models fitted only in the other blocks (four folds).

In [4]:
rows = []
for c in checks:
    for check, name in (("gap", "Gap of one year"), ("blocks", "Fitted in other blocks")):
        a = {k: sum(v) / 3 for k, v in c[check].items()}
        rows.append({"Window": c["split"], "Check": name, **{k: round(v, 3) for k, v in a.items()},
                     "Gain": round(a["Improved + TESSERA"] - a["Improved model"], 3)})
pd.DataFrame(rows)

,Window,Check,Improved model,Improved + TESSERA,TESSERA alone,Gain
0,2018–2020 → 2021–2023,Gap of one year,0.647,0.799,0.790,0.152
1,2018–2020 → 2021–2023,Fitted in other blocks,0.614,0.776,0.766,0.161
2,2019–2021 → 2022–2024,Gap of one year,0.655,0.770,0.758,0.115
3,2019–2021 → 2022–2024,Fitted in other blocks,0.615,0.750,0.739,0.136


Both checks keep the gain (+0.12 to +0.16). Fitted in other blocks, every model loses a little, the improved model as much as the others.

## 3 · Is it only the land cover?

Esri's yearly map, which the platform already reads, says what covers each cell. Its class shares at the window's start were added to the improved model on the same cells and draws.

In [5]:
rows = []
for e in esri:
    rows.append({"Window": e["split"], **{k: round(sum(v) / 3, 3) for k, v in e.items() if k != "split"}})
pd.DataFrame(rows)

,Window,Improved model,Improved + Esri shares,Improved + TESSERA,Improved + Esri + TESSERA
0,2018–2020 → 2021–2023,0.657,0.710,0.810,0.808
1,2019–2021 → 2022–2024,0.670,0.714,0.786,0.786


Esri's shares help (+0.04 to +0.05) and add nothing once TESSERA is in. TESSERA carries what the land cover carries and more: a year of radar and optical behaviour of each pixel, which separates closed canopy from fallow and partly cleared forest inside cells that Hansen still counts as forest.

## 4 · What it would take on the platform

* **Data.** 2,048 TESSERA tiles hold Guinea's forest (12.6 million cells of 100 m). Each tile takes 20–45 s to read here, about 20 hours a year of embeddings if read one tile at a time (about 9 hours with four tiles read at a time, section 5). Only the cell values need keeping: 16 principal components (88 % of the variance, the same AUC as the 128 numbers) over the forest cells are about 0.4 GB a year.
* **Years.** The platform calibrates on the latest window and maps at the latest year, both after 2017, so TESSERA's start limits the backtest windows only: the first window of notebooks 03–10 (2014–2016) cannot use it.
* **Verra's tests.** VT0007's fit and prediction errors are counted in blocks of about 100,000 ha over the whole country, so they need the national embeddings: read for one window in section 5.
* **New years.** Each year's embeddings must be released before the platform can map with them (2025 was available in November 2026).

## 5 · The national test

The blocks cannot give Verra's errors, so the embeddings were read over all 2,048 tiles with forest for the third window (`data_100m_tools/tessera_national.py`: 2018 and 2021, about 9 hours a year with four tiles at a time, 0.4 GB a year kept). Each forest cell keeps 16 principal components of its mean embedding, from a rotation fitted once on the forest cells of the 24 sample blocks (85 % of the variance). The national file matches the block files (correlation 0.99999); 799 of 12.6 million cells have no data and take the components' mean.

The test (`scripts/tessera_national_test.py`, run `outputs/runs/11_tessera_national_2018_2021_2024.json`) repeats notebook 08's national comparison with and without TESSERA: the same draw of training cells over the whole country, seeds 42, 1 and 2, AUC over the 30 m pixels, VT0007's fit and prediction MedAE, for the improved model alone and for the adopted risk map, its rank average with riskmapjnr's local rate (option A, notebook 10).

**Rule, written in the script before the run:** the second window (2017, 2020) is read only if, for both the model alone and the rank average, TESSERA raises the AUC in every seed by more than the noise between runs (0.003) and lowers the mean prediction error by more than the spread between seeds of the map without it.

In [6]:
nat = json.loads((RUNS / "11_tessera_national_2018_2021_2024.json").read_text(encoding="utf-8"))
pd.DataFrame([{"Risk map": k, "AUC": round(sum(v["aucs"]) / 3, 3),
               "Fit MedAE (ha)": round(sum(v["fit"]) / 3), "Prediction MedAE (ha)": round(sum(v["prediction"]) / 3),
               "Prediction, seeds": ", ".join(f"{x:.0f}" for x in v["prediction"])} for k, v in nat["variants"].items()])

,Risk map,AUC,Fit MedAE (ha),Prediction MedAE (ha),"Prediction, seeds"
0,Improved model,0.737,128,128,"137, 133, 113"
1,Rank average (A): improved model + riskmapjnr,0.737,118,113,"122, 113, 103"
2,Improved + TESSERA,0.805,183,196,"179, 191, 219"
3,Rank average (A): improved + TESSERA + riskmapjnr,0.784,120,141,"146, 134, 144"


In [7]:
pd.DataFrame([{"With TESSERA": k, "AUC gain, seeds": ", ".join(f"{g:+.3f}" for g in d["auc_gain_per_seed"]),
               "Prediction error change (ha)": round(-d["prediction_gain_ha"]), "Seed spread without (ha)": round(d["seed_range_ha"]),
               "AUC clear": d["auc_clear"], "Verra clear": d["prediction_clear"]} for k, d in nat["decision"].items()])

,With TESSERA,"AUC gain, seeds",Prediction error change (ha),Seed spread without (ha),AUC clear,Verra clear
0,Improved + TESSERA,"+0.069, +0.068, +0.068",69,24,True,False
1,Rank average (A): improved + TESSERA + riskmapjnr,"+0.046, +0.048, +0.047",28,20,True,False


The rows without TESSERA reproduce notebook 08's national results for this window (AUC 0.737; rank average fit 118 ha, prediction 113 ha). With TESSERA the AUC rises in every seed, by +0.068 for the model alone and +0.047 for the rank average, twenty times the noise between runs. Verra's errors move the other way: the model alone misses by 196 ha instead of 128, the rank average by 141 ha instead of 113, beyond the spread between seeds (24 and 20 ha). TESSERA's first two components rank second and third in the forest's importance, after the 5 km loss rate.

The two scores answer different questions. AUC asks whether the cells that lost forest were ranked above those that kept it; it is blind to how much loss each place receives and to where the errors fall (Lobo et al. 2008; Pontius & Parmentier 2014). Verra's test asks how close the predicted hectares are to the real ones in blocks of about 100,000 ha, which is what a jurisdictional baseline allocates, and a projection that ranks well can still place the change worse than a simple benchmark (Sloan & Pelletier 2012). One model serves the whole module, so a change must pass both, and Verra's prediction test decides. By the rule above the second window was not read.

## What this means

* **TESSERA gives the largest gain in AUC found for the risk model:** +0.11 to +0.15 AUC in the forest blocks, in both windows that can be tested, against ±0.002 for the candidates of notebook 05. It holds with a one-year gap and when blocks are scored by models fitted elsewhere.
* **It is more than land cover:** Esri's class shares give a third of the gain and nothing on top of TESSERA.
* **Nationally the ranking gain holds, Verra's score does not.** Over all of Guinea, TESSERA raises the AUC by +0.068 (model alone) and +0.047 (rank average) in every seed, and worsens Verra's prediction error by 69 ha and 28 ha, more than the spread between seeds.
* **Not adopted for the risk map.** The risk map's task under VT0007 is to place the expected loss in hectares, and Verra's prediction test is the score that decides (rule set before the run). The second window was therefore not read.

### References

* Feng, Z., Atzberger, C., Jaffer, S., Knezevic, J., Sormunen, S., Young, R., Lisaius, M.C., Immitzer, M., Jackson, T., Ball, J., et al. (2026). TESSERA: Temporal embeddings of surface spectra for earth representation and analysis. *CVPR 2026*, 34818–34831. arXiv:2506.20380.
* Hansen, M.C. et al. (2013). High-resolution global maps of 21st-century forest cover change. *Science* 342, 850–853.
* Lobo, J.M., Jiménez-Valverde, A. & Real, R. (2008). AUC: a misleading measure of the performance of predictive distribution models. *Global Ecology and Biogeography* 17, 145–151.
* Pontius, R.G. Jr. & Parmentier, B. (2014). Recommendations for using the relative operating characteristic (ROC). *Landscape Ecology* 29, 367–382.
* Sloan, S. & Pelletier, J. (2012). How accurately may we project tropical forest-cover change? A validation of a forward-looking baseline for REDD. *Global Environmental Change* 22, 440–453.
* Verra (2024). *VT0007 Unplanned Deforestation Allocation (UDef-A), v1.0*.